In [1]:
import pandas as pd
import numpy as np
import random

random.seed(42)
np.random.seed(42)

print("Environment Ready")

Environment Ready


In [2]:
customers = pd.read_csv("../data/raw/DimCustomer.csv")

accounts = pd.read_csv("../data/raw/DimAccount.csv")

branches = pd.read_csv("../data/raw/DimBranch.csv")

print(len(customers))
print(len(accounts))
print(len(branches))

50000
75000
100


In [3]:
customer_branch = (
    accounts
    .sort_values("AccountID")
    .groupby("CustomerID", as_index=False)
    .first()[["CustomerID", "BranchID"]]
)

customer_branch.head()

,CustomerID,BranchID
0,1,6
1,2,30
2,3,28
3,4,47
4,5,30


In [6]:
loan_base = customers.merge(
    customer_branch,
    on="CustomerID",
    how="inner"
)

print(len(loan_base))

38909


In [7]:
loan_customers = loan_base.sample(
    n=30000,
    random_state=42
).copy()

loan_customers.reset_index(drop=True, inplace=True)

print(len(loan_customers))

30000


In [8]:
loan_types = np.random.choice(
    [1, 2, 3, 4],
    size=30000,
    p=[
        0.45,   # Mortgage
        0.25,   # Auto
        0.20,   # Personal
        0.10    # Commercial
    ]
)

loan_customers["LoanTypeID"] = loan_types

In [9]:
credit_tiers = np.random.choice(
    [1,2,3,4,5],
    size=30000,
    p=[
        0.08,
        0.15,
        0.37,
        0.25,
        0.15
    ]
)

loan_customers["CreditScoreTierID"] = credit_tiers

In [10]:
origination_dates = pd.to_datetime(
    np.random.choice(
        pd.date_range(
            "2020-01-01",
            "2026-07-31"
        ),
        size=30000
    )
)

loan_customers["OriginationDate"] = origination_dates

In [11]:
loan_customers["OriginationDateKey"] = (
    loan_customers["OriginationDate"]
    .dt.strftime("%Y%m%d")
    .astype(int)
)

In [12]:
def generate_loan_amount(loan_type):

    if loan_type == 1:
        return np.random.randint(150000,800000)

    elif loan_type == 2:
        return np.random.randint(15000,70000)

    elif loan_type == 3:
        return np.random.randint(3000,50000)

    else:
        return np.random.randint(200000,3000000)

In [13]:
loan_customers["LoanAmount"] = loan_customers[
    "LoanTypeID"
].apply(generate_loan_amount)

In [14]:
loan_customers[
    ["LoanTypeID","LoanAmount"]
].head()

,LoanTypeID,LoanAmount
0,1,569550
1,4,2029180
2,3,47445
3,2,45667
4,1,255557


In [15]:
def generate_interest_rate(tier):

    if tier == 5:      # Excellent
        return round(np.random.uniform(3.0, 4.5), 2)

    elif tier == 4:    # Very Good
        return round(np.random.uniform(4.0, 5.5), 2)

    elif tier == 3:    # Good
        return round(np.random.uniform(5.0, 7.0), 2)

    elif tier == 2:    # Fair
        return round(np.random.uniform(7.0, 10.0), 2)

    else:              # Poor
        return round(np.random.uniform(10.0, 16.0), 2)

In [16]:
loan_customers["InterestRate"] = loan_customers[
    "CreditScoreTierID"
].apply(generate_interest_rate)

In [17]:
loan_customers[
    ["CreditScoreTierID", "InterestRate"]
].head(10)

,CreditScoreTierID,InterestRate
0,4,4.57
1,3,5.28
2,5,4.18
3,2,8.54
4,3,5.22
5,4,4.78
6,4,4.49
7,3,6.00
8,4,4.39
9,3,6.51


In [18]:
loan_customers["TermMonths"] = loan_customers["LoanTypeID"].map({
    1: 360,
    2: 72,
    3: 48,
    4: 180
})

In [19]:
loan_customers[
    ["LoanTypeID", "TermMonths"]
].head()

,LoanTypeID,TermMonths
0,1,360
1,4,180
2,3,48
3,2,72
4,1,360


In [20]:
def monthly_payment(principal, annual_rate, months):

    r = annual_rate / 100 / 12

    if r == 0:
        return principal / months

    payment = (
        principal * r *
        (1 + r) ** months
    ) / (
        (1 + r) ** months - 1
    )

    return round(payment, 2)

In [21]:
loan_customers["MonthlyPayment"] = loan_customers.apply(
    lambda row: monthly_payment(
        row["LoanAmount"],
        row["InterestRate"],
        row["TermMonths"]
    ),
    axis=1
)

In [22]:
loan_customers[
    [
        "LoanAmount",
        "InterestRate",
        "TermMonths",
        "MonthlyPayment"
    ]
].head()

,LoanAmount,InterestRate,TermMonths,MonthlyPayment
0,569550,4.57,360,2909.56
1,2029180,5.28,180,16344.15
2,47445,4.18,48,1075.09
3,45667,8.54,72,812.79
4,255557,5.22,360,1406.45


In [23]:
remaining_factor = np.random.uniform(
    0.20,
    1.00,
    size=len(loan_customers)
)

loan_customers["OutstandingBalance"] = np.round(
    loan_customers["LoanAmount"] * remaining_factor,
    2
)

In [24]:
loan_customers[
    [
        "LoanAmount",
        "OutstandingBalance"
    ]
].head()

,LoanAmount,OutstandingBalance
0,569550,141092.89
1,2029180,1164740.80
2,47445,30049.92
3,45667,39441.38
4,255557,194450.44


In [25]:
loan_customers["DaysPastDue"] = np.random.choice(
    [0, 30, 60, 90, 120, 180],
    size=len(loan_customers),
    p=[
        0.88,
        0.05,
        0.03,
        0.02,
        0.01,
        0.01
    ]
)

In [26]:
loan_customers["DaysPastDue"].value_counts()

DaysPastDue
0      26362
30      1493
60       910
90       646
120      301
180      288
Name: count, dtype: int64

In [28]:
loan_customers["LoanStatus"] = np.select(
    [
        loan_customers["DaysPastDue"] == 0,
        loan_customers["DaysPastDue"].isin([30, 60]),
        loan_customers["DaysPastDue"] >= 90
    ],
    [
        "Current",
        "Delinquent",
        "Non-Performing"
    ],
    default="Current"
)

In [29]:
loan_customers["LoanStatus"].value_counts()

LoanStatus
Current           26362
Delinquent         2403
Non-Performing     1235
Name: count, dtype: int64

In [30]:
loan_customers["IsNonPerforming"] = (
    loan_customers["DaysPastDue"] >= 90
).astype(int)

In [43]:
loan_customers[
    [
        "DaysPastDue",
        "IsNonPerforming"
    ]
].head(10)

,DaysPastDue,IsNonPerforming
0,0,0
1,0,0
2,0,0
3,0,0
4,0,0
5,0,0
6,30,0
7,0,0
8,0,0
9,0,0


In [32]:
loan_customers["MaturityDate"] = (
    loan_customers["OriginationDate"] +
    pd.to_timedelta(
        loan_customers["TermMonths"] * 30,
        unit="D"
    )
)

In [33]:
loan_customers["MaturityDateKey"] = (
    loan_customers["MaturityDate"]
    .dt.strftime("%Y%m%d")
    .astype(int)
)

In [34]:
fact_loans = loan_customers[
    [
        "CustomerID",
        "BranchID",
        "LoanTypeID",
        "CreditScoreTierID",
        "OriginationDateKey",
        "MaturityDateKey",
        "LoanAmount",
        "OutstandingBalance",
        "InterestRate",
        "TermMonths",
        "MonthlyPayment",
        "DaysPastDue",
        "LoanStatus",
        "IsNonPerforming"
    ]
].copy()

In [35]:
fact_loans.head()

,CustomerID,BranchID,LoanTypeID,CreditScoreTierID,OriginationDateKey,MaturityDateKey,LoanAmount,OutstandingBalance,InterestRate,TermMonths,MonthlyPayment,DaysPastDue,LoanStatus,IsNonPerforming
0,33882,75,1,4,20250201,20540828,569550,141092.89,4.57,360,2909.56,0,Current,0
1,48787,28,4,3,20230222,20371205,2029180,1164740.80,5.28,180,16344.15,0,Current,0
2,6974,54,3,5,20260613,20300523,47445,30049.92,4.18,48,1075.09,0,Current,0
3,32116,14,2,2,20230506,20290404,45667,39441.38,8.54,72,812.79,0,Current,0
4,23973,10,1,3,20241008,20540504,255557,194450.44,5.22,360,1406.45,0,Current,0


In [36]:
print(f"Rows: {len(fact_loans):,}")

Rows: 30,000


In [37]:
fact_loans.isnull().sum()

CustomerID            0
BranchID              0
LoanTypeID            0
CreditScoreTierID     0
OriginationDateKey    0
MaturityDateKey       0
LoanAmount            0
OutstandingBalance    0
InterestRate          0
TermMonths            0
MonthlyPayment        0
DaysPastDue           0
LoanStatus            0
IsNonPerforming       0
dtype: int64

In [38]:
(
    fact_loans["OutstandingBalance"]
    > fact_loans["LoanAmount"]
).sum()

np.int64(0)

In [39]:
fact_loans["IsNonPerforming"].value_counts()

IsNonPerforming
0    28765
1     1235
Name: count, dtype: int64

In [40]:
output_path = "../data/raw/FactLoans.csv"

fact_loans.to_csv(
    output_path,
    index=False
)

print(f"Saved {len(fact_loans):,} loans to {output_path}")

Saved 30,000 loans to ../data/raw/FactLoans.csv
